# Judging a reply with Jev 🧪

TypeSafe's [Jev](https://docs.typesafe.ai/concepts/system-one) answers typed questions
about a piece of evidence and returns probabilities. It does not generate prose.

This notebook builds an evaluation rubric one question at a time, sends it in a single
request, and reads the answers. Notebooks 2 and 3 reuse the finished rubric to score
live spans and to drive a Datadog experiment.

You only need a `TYPESAFE_API_KEY` for this notebook. Nothing here talks to Datadog or
OpenAI yet.

## Setup

Install the dependencies and create a `.env` file next to this notebook:

```bash
pip install -r requirements.txt
```

```bash
TYPESAFE_API_KEY=<your key>
```

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

from typesafe_sdk import Choice, Noul, NoulCriteria, Score, TypeSafeClient

from vega_air import TICKETS, retrieve

## The evidence and the three question types

Everything Jev looks at is called the **state**. It can be a string, or a JSON object
whose keys you can refer to by name from inside a question. The rubric below uses three
keys: the customer's `question`, the retrieved `policy_context`, and the agent's `reply`.

There are three question types. Each one returns a different shape:

| Type | Returns |
| --- | --- |
| [Noul](https://docs.typesafe.ai/primitives/noul) | `noul`, the probability that a yes/no proposition is true |
| [Choice](https://docs.typesafe.ai/primitives/choice) | the selected option, a probability per option, and a confidence |
| [Score](https://docs.typesafe.ai/primitives/score) | a probability-weighted value over ordered levels, the level distribution, a confidence, and the rubric legend |

Nouls carry no confidence. [Confidence](https://docs.typesafe.ai/confidence) is derived
from the shape of a distribution, and a Noul does not return one.

## A Noul: is the reply grounded?

The agent answers from retrieved policy excerpts. The failure worth catching is a reply
that invents a fee or a deadline that appears nowhere in those excerpts.

Three fields do the work. `instructions` states the question and scopes it. `criteria`
spells out what counts as yes and what counts as no. `inspect` names the part of the
state being judged.

Jev reads all of this literally, which is the point. Vague scoping produces answers that
are correct against what you wrote and wrong against what you meant.

In [ ]:
grounded = Noul(
    instructions={
        "question": (
            "Is every factual claim about Vega Air policy in `reply` stated in, or "
            "directly restated from, `policy_context`?"
        ),
        "inspect": "reply",
        "scope": [
            "Only policy claims count: fees, amounts, deadlines, weight limits, eligibility.",
            "Ignore greetings, apologies, and offers to hand off to a human agent.",
            "A reply that states no policy claims at all is grounded.",
        ],
    },
    criteria=NoulCriteria(
        true="Every policy claim in `reply` appears in `policy_context`.",
        false=(
            "At least one policy claim in `reply` is absent from `policy_context`, "
            "contradicts it, or changes a number, fee, or deadline."
        ),
    ),
)

Two more Nouls cover the other things a good reply has to do: supply the information
that was asked for, or route the customer to a human.

Keeping these separate is deliberate. One question that weighs three criteria at once
gives you one number you cannot decompose. Three questions give you three numbers you
can combine however you like, and re-combine later without re-running anything.

In [ ]:
answers_question = Noul(
    instructions={
        "question": "Does `reply` give the customer the specific information asked for in `question`?",
        "inspect": "reply",
        "scope": [
            "Judge whether the information was supplied, not whether it is correct.",
            "A reply that only promises to look into it has not supplied the information.",
        ],
    },
    criteria=NoulCriteria(
        true="`reply` states the fee, rule, amount, or answer that `question` asked for.",
        false="`reply` deflects, asks a counter-question, or only describes next steps.",
    ),
)

offers_handoff = Noul(
    instructions={
        "question": "Does `reply` offer to transfer the customer to a human agent or another team?",
        "inspect": "reply",
    },
    criteria=NoulCriteria(
        true="`reply` offers a human agent, a specialist team, or a support contact.",
        false="`reply` makes no such offer.",
    ),
)

## A Choice: what went wrong?

A Choice picks one option from a set you define. Note the `unclear` option: a Choice
always returns whichever option has the highest probability, so Jev never declines to
answer. If your evaluation needs a way to say "cannot judge this one", that outcome has
to exist in the criteria. TypeSafe's
[self-consistency cookbook](https://docs.typesafe.ai/cookbooks/consistency_choice_cookbook)
uses the same trick for moderation.

In [ ]:
failure_mode = Choice(
    instructions={
        "question": "What is the single biggest problem with `reply`?",
        "scope": (
            "Pick `none` when the reply is fine. Pick `unclear` only when the reply "
            "is too short or too garbled to judge."
        ),
    },
    criteria={
        "none": "The reply is accurate, on-policy, and useful.",
        "unsupported_claim": "The reply states a fee, rule, or number that is not in `policy_context`.",
        "missed_handoff": (
            "`policy_context` does not cover the question and the reply neither says so "
            "nor offers a human agent."
        ),
        "partial_answer": "The reply covers part of the question and silently drops the rest.",
        "unsafe_request": (
            "The reply complies with a request for personal data or something outside "
            "support scope."
        ),
        "unclear": "The reply is too short or too garbled to judge.",
    },
)

## A Score: how bad is it for the customer?

A Score rates the state against ordered levels. The position of each description is its
value, starting at zero. The answer is the probability-weighted average of those levels,
so it can land between them.

In [ ]:
customer_impact = Score(
    instructions={
        "question": "If this reply were sent as-is, how badly would it affect the customer?",
        "scope": "Rate the consequence for the customer, not the tone of the reply.",
    },
    criteria=[
        "No harm. The customer gets what they need.",
        "Mild friction. The customer must ask again or look elsewhere.",
        "Real cost. The customer acts on wrong information or is stranded without a route forward.",
        "Serious harm. The customer loses money, misses travel, or their privacy is breached.",
    ],
)

QUESTIONS = {
    "grounded": grounded,
    "answers_question": answers_question,
    "offers_handoff": offers_handoff,
    "failure_mode": failure_mode,
    "customer_impact": customer_impact,
}

list(QUESTIONS)

## One request, five answers

Questions in the same request are answered independently against the same state, so one
cannot contaminate another. The state is also charged once rather than once per question,
which is why batching a rubric is much cheaper than one request per criterion.

Pin the model. `jev-latest` and `jev-preview` both point at a specific version today, and
both move when a release ships. Thresholds are tuned against one model, so ask for a
versioned ID and record what answered.

The reply below is a fixed example so this notebook runs without OpenAI. Notebook 2
generates replies with the real agent.

In [ ]:
JEV_MODEL = "jev-1.13.0"

ticket = next(t for t in TICKETS if t["id"] == "t-1005")
policy_context = retrieve(ticket["context_keys"])
reply = (
    "Thanks for reaching out. The excerpts I have cover delay compensation under EU261, "
    "but they do not set out what Vega Air pays when a flight is cancelled outright. "
    "I would rather not quote a figure that might be wrong, so let me put you through to "
    "a colleague who can check your booking and confirm."
)

print("question:", ticket["question"])
print()
print("policy_context:", policy_context)

In [ ]:
client = TypeSafeClient(model=JEV_MODEL)

result = client.system_one(
    state={"question": ticket["question"], "policy_context": policy_context, "reply": reply},
    questions=QUESTIONS,
)

print("answered by:", result.model)
print("usage:", result.usage.input_tokens, "input tokens,", result.usage.output_tokens, "output tokens")

## Reading the answers

Answers come back keyed by the names you chose. The SDK groups them by type, so
`result.nouls`, `result.choices`, and `result.scores` each hold the matching subset.

In [ ]:
for name, answer in result.nouls.items():
    print(f"{name:20} P(yes) = {answer.noul:.2f}")

In [ ]:
choice = result.choices["failure_mode"]
print("picked:    ", choice.choice)
print("confidence:", round(choice.confidence, 2))
print()
for option, p in sorted(choice.probabilities.items(), key=lambda kv: -kv[1]):
    bar = "#" * round(p * 40)
    print(f"  {option:20} {p:.2f} {bar}")

Look at the spread rather than only the label. When the top two options are close, the
label is a coin flip dressed up as a decision, and the confidence value says so. That is
a useful trigger for routing a trace to a human reviewer, and it is information you throw
away if you keep only the winning string.

In [ ]:
impact = result.scores["customer_impact"]
print(f"score: {impact.score:.2f}  confidence: {impact.confidence:.2f}")
print()
for level in sorted(impact.probabilities):
    p = impact.probabilities[level]
    print(f"  {level}  {p:.2f}  {impact.legend[level]}")

## Combining the answers in code

Jev is not a calculator, and it does not know your risk tolerance. The composite verdict
and the thresholds behind it belong in application code, where they are easy to read and
easy to retune without touching the rubric or re-running the judge.

In [ ]:
GROUNDED_THRESHOLD = 0.70
ANSWERED_THRESHOLD = 0.60
HANDOFF_THRESHOLD = 0.60

g = result.nouls["grounded"].noul
a = result.nouls["answers_question"].noul
h = result.nouls["offers_handoff"].noul

# Handled correctly means grounded, and either answered or routed to a human.
handled = g >= GROUNDED_THRESHOLD and (a >= ANSWERED_THRESHOLD or h >= HANDOFF_THRESHOLD)

print(f"grounded={g:.2f} answered={a:.2f} handoff={h:.2f} -> handled_correctly={handled}")

## Where this goes next

The finished rubric, the thresholds, and the `verdict()` helper live in
[`jev_rubric.py`](./jev_rubric.py) so the next two notebooks import one definition rather
than keeping two copies in sync. The check below confirms this notebook and that module
still describe the same five questions.

In [ ]:
from jev_rubric import QUESTIONS as PACKAGED

assert QUESTIONS.keys() == PACKAGED.keys(), "notebook rubric and jev_rubric.py have drifted"
print("in sync:", sorted(PACKAGED))

Next:

- **[2-online-evals.ipynb](./2-online-evals.ipynb)** scores live Datadog spans with this rubric.
- **[3-experiments.ipynb](./3-experiments.ipynb)** runs it over a dataset inside a Datadog experiment.

Before you trust a rubric on real traffic, run it over twenty cases you labeled yourself
and read every disagreement. Jev answers the question you wrote, so a disagreement is
often a scoping bug in the criteria rather than a mistake by the model. TypeSafe lists the
failure modes worth knowing about on its
[jaggedness page](https://docs.typesafe.ai/model-jaggedness/jev-1.13).